# Simple Vector Search with Existing ChromaDB Vectors

This notebook queries vectors that already exist in your ChromaDB collection.

Flow:
1. Connect to the existing local Chroma database.
2. Embed a user query with the same model used during ingestion (`text-embedding-3-small`).
3. Run `collection.query(...)` and return top matches.


In [6]:
from pathlib import Path
import json
import os

import chromadb
from dotenv import load_dotenv
from openai import OpenAI


In [7]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
CHROMA_PATH = PROJECT_ROOT / 'backend' / 'data' / 'chroma_db'
COLLECTION_NAME = 'estate_documents'
EMBEDDING_MODEL_NAME = 'text-embedding-3-small'
TOP_K = 5

load_dotenv(PROJECT_ROOT / '.env', override=False)

if not (os.getenv('OPENAI_API_KEY') or '').strip():
    raise ValueError('OPENAI_API_KEY is missing. Add it to /workspace/.env or your environment.')

print(f'Project root: {PROJECT_ROOT}')
print(f'Chroma path: {CHROMA_PATH}')
print(f'Collection: {COLLECTION_NAME}')


Project root: /workspace
Chroma path: /workspace/backend/data/chroma_db
Collection: estate_documents


In [8]:
if not CHROMA_PATH.exists():
    raise FileNotFoundError(f'Chroma path does not exist: {CHROMA_PATH}')

client = chromadb.PersistentClient(path=str(CHROMA_PATH))
collection = client.get_or_create_collection(name=COLLECTION_NAME)

print(f'Collection count: {collection.count()} vectors')


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Collection count: 93 vectors


In [9]:
openai_client = OpenAI()

def embed_query(query: str) -> list[float]:
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL_NAME,
        input=[query],
    )
    return response.data[0].embedding

def search_chroma(query: str, top_k: int = TOP_K, where: dict | None = None):
    query_embedding = embed_query(query)

    query_kwargs = {
        'query_embeddings': [query_embedding],
        'n_results': top_k,
        'include': ['documents', 'metadatas', 'distances'],
    }
    if where is not None:
        query_kwargs['where'] = where

    result = collection.query(**query_kwargs)

    ids = result.get('ids', [[]])[0]
    documents = result.get('documents', [[]])[0]
    metadatas = result.get('metadatas', [[]])[0]
    distances = result.get('distances', [[]])[0]

    rows = []
    for idx, doc_id in enumerate(ids):
        rows.append({
            'id': doc_id,
            'document': documents[idx],
            'metadata': metadatas[idx] or {},
            'distance': distances[idx],
        })

    return rows


In [10]:
# Change this query text to test different searches.
user_query = 'What does the document say about mortgage terms?'

# Optional metadata filter examples:
# where_filter = {'document_type': 'mortgage_deed'}
# where_filter = {'has_currency_amount': 1}
where_filter = None

results = search_chroma(user_query, top_k=5, where=where_filter)
print(f'Query: {user_query}')
print(f'Where filter: {where_filter}\n')

for rank, row in enumerate(results, start=1):
    metadata = row['metadata']

    people = metadata.get('person_names_mentioned', '[]')
    years = metadata.get('mentioned_years', '[]')
    dates = metadata.get('mentioned_dates', '[]')
    amounts = metadata.get('amounts_eur', '[]')
    roles = metadata.get('legal_roles_mentioned', '[]')
    try:
        people = json.loads(people)
    except Exception:
        people = []
    try:
        years = json.loads(years)
    except Exception:
        years = []
    try:
        dates = json.loads(dates)
    except Exception:
        dates = []
    try:
        amounts = json.loads(amounts)
    except Exception:
        amounts = []
    try:
        roles = json.loads(roles)
    except Exception:
        roles = []

    excerpt = (row['document'] or '').replace('\n', ' ')[:220]

    print(f"{rank}. id={row['id']} distance={row['distance']:.6f}")
    print(f"   document_id={metadata.get('document_id')} page={metadata.get('page_number')}")
    print(f"   people={people}")
    print(f"   years={years} dates={dates}")
    print(f"   amounts_eur={amounts} roles={roles}")
    print(f"   text={excerpt}...\n")


Query: What does the document say about mortgage terms?
Where filter: None

1. id=mortgage_014-p001-c0002 distance=0.912302
   document_id=mortgage_014 page=1
   people=[]
   years=[] dates=[]
   amounts_eur=[] roles=['borrower', 'lender']
   text=NOW, THEREFORE, the parties agree as follows:  ---  **MORTGAGE GRANT**  1. The Borrowers hereby grant to the Lender a first-ranking mortgage on the Secured Property as collateral for the repayment of the Loan Amount and ...

2. id=mortgage_014-p001-c0000 distance=0.970853
   document_id=mortgage_014 page=1
   people=['Annelies De Smet', 'Sarah Janssen-Peeters', 'Thomas Janssen']
   years=[1972, 1975, 2024] dates=['4 February 1975', '6 February 2024', '9 November 1972']
   amounts_eur=[] roles=['borrower', 'lender']
   text=**MORTGAGE DEED (Hypotheekakte)**  Document ID: mortgage_014 Date of Deed: 6 February 2024 Place of Execution: Bruges, Belgium Notary: Meester Annelies De Smet, Bruges  ---  **PARTIES APPEARING**  **Lenders**: 1. **Belfi...